# EDA — Mix energetico in Europa: nucleare, rinnovabili e fossili

**Obiettivo del notebook**

Esplorare come è cambiato nel tempo il mix di fonti usate per produrre elettricità nei paesi europei,
con un focus particolare sul confronto tra **nucleare**, **rinnovabili** e **fonti fossili**.

## 1. Fonte dei dati

I dati usati in questo notebook provengono dal dataset **Energy Data** di
[**Our World in Data (OWID)**](https://github.com/owid/energy-data), un progetto di ricerca open-access
dell'Università di Oxford.

- OWID non raccoglie i dati in prima persona, ma li **aggrega e armonizza** da fonti istituzionali
  riconosciute a livello internazionale, principalmente:
  - **Ember** — *Yearly Electricity Data*, dati elettrici annuali (la fonte principale per l'elettricità dal 2000 in poi,
    e dal 1990 per i paesi europei)
  - **Energy Institute** — *Statistical Review of World Energy* (fonte storica, usata per gli anni precedenti)
  - Dati di popolazione e PIL da fonti demografiche/economiche standard usate da OWID
- Il dataset è **aggiornato regolarmente** (ultimo aggiornamento verificato: 2026) ed è openly licensed
  (**CC BY** — riutilizzabile citando la fonte)
- È lo stesso dataset usato per generare i grafici pubblicati su https://ourworldindata.org/energy

**Link diretti:**
- Repository GitHub: https://github.com/owid/energy-data
- File CSV completo: `owid-energy-data.csv` (nella root del repository)
- Codebook (descrizione di ogni colonna): `owid-energy-codebook.csv`
- Pagina di presentazione: https://ourworldindata.org/energy

**Citazione consigliata** (come richiesto da OWID per l'uso dei dati):
> Ember (2026); Energy Institute - Statistical Review of World Energy (2025) — with major processing by Our World in Data.

In [14]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path.cwd()
df = pd.read_csv(DATA_DIR / "data" / "owid-energy-data.csv")

print(df.shape)
df.head()

(23377, 130)


,country,year,iso_code,population,gdp,biofuel_cons_change_pct,biofuel_cons_change_twh,biofuel_cons_per_capita,biofuel_consumption,biofuel_elec_per_capita,...,solar_share_elec,solar_share_energy,wind_cons_change_pct,wind_cons_change_twh,wind_consumption,wind_elec_per_capita,wind_electricity,wind_energy_per_capita,wind_share_elec,wind_share_energy
0,ASEAN (Ember),2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
1,ASEAN (Ember),2001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
2,ASEAN (Ember),2002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
3,ASEAN (Ember),2003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
4,ASEAN (Ember),2004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN


## 2. Struttura del dataset

Il file è in formato **"long"**: una riga per ogni combinazione di *paese* e *anno*.
Prima di tutto controlliamo dimensioni, periodo temporale coperto e quante "entità" (paesi/aggregati)
sono presenti.


In [15]:
print(f"Righe: {df.shape[0]:,}")
print(f"Colonne: {df.shape[1]}")
print(f"Periodo: {df['year'].min()} - {df['year'].max()}")
print(f"Entità (paesi + aggregati): {df['country'].nunique()}")


Righe: 23,377
Colonne: 130
Periodo: 1900 - 2025
Entità (paesi + aggregati): 314


**Nota importante:** la colonna `country` non contiene *solo* paesi. OWID include anche
**aggregati** (continenti, "World", gruppi di reddito, organizzazioni come "EU (Ember)", "G7 (Ember)" ecc.)
per permettere confronti a livello macro nei propri grafici.

Questi aggregati **non hanno un `iso_code`** (il codice ISO a 3 lettere dei paesi), quindi possiamo
usare questo campo per individuarli ed escluderli quando vogliamo lavorare solo su paesi reali.


In [16]:
# Quante "entità" non sono paesi veri (non hanno iso_code)?
aggregati = sorted(df.loc[df["iso_code"].isna(), "country"].unique())
print(f"Aggregati/non-paesi individuati: {len(aggregati)}")
aggregati[:15]  # primi esempi


Aggregati/non-paesi individuati: 94


['ASEAN (Ember)',
 'Africa',
 'Africa (EI)',
 'Africa (EIA)',
 'Africa (Ember)',
 'Africa (Shift)',
 'Asia',
 'Asia (Ember)',
 'Asia Pacific (EI)',
 'Asia and Oceania (EIA)',
 'Asia and Oceania (Shift)',
 'Australia and New Zealand (EIA)',
 'CIS (EI)',
 'Central America (EI)',
 'Central and South America (EIA)']